# 03 — Sistemas de Ecuaciones Lineales

**Objetivo:** Representar y resolver sistemas $A\mathbf{x} = \mathbf{b}$ con eliminación gaussiana y con NumPy. Entender los tres casos posibles.

## Contexto matemático

Un sistema de $m$ ecuaciones con $n$ incógnitas:

$$\begin{cases}
a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n = b_1 \\
a_{21}x_1 + a_{22}x_2 + \cdots + a_{2n}x_n = b_2 \\
\quad\vdots \\
a_{m1}x_1 + a_{m2}x_2 + \cdots + a_{mn}x_n = b_m
\end{cases}$$

En forma matricial: $A\mathbf{x} = \mathbf{b}$ con $A \in \mathbb{R}^{m\times n}$, $\mathbf{x} \in \mathbb{R}^n$, $\mathbf{b} \in \mathbb{R}^m$.

**Interpretación geométrica:** cada ecuación define un hiperplano en $\mathbb{R}^n$. La solución es la intersección de todos los hiperplanos.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Contexto analytics: tasas de conversión base por canal

Supón que observamos el total de activaciones en 3 períodos, y cada período mezcla 3 canales con distintos volúmenes conocidos. Queremos encontrar la tasa de conversión base $x_i$ de cada canal.

$$\underbrace{\begin{pmatrix} 1200 & 800 & 500 \\ 900 & 1100 & 700 \\ 600 & 950 & 1300 \end{pmatrix}}_{A\text{ (usuarios por canal por período)}}\mathbf{x} = \underbrace{\begin{pmatrix} 148 \\ 162 \\ 201 \end{pmatrix}}_{\mathbf{b}\text{ (activaciones totales)}}$$

In [ ]:
A = np.array([
    [1200., 800., 500.],
    [ 900., 1100., 700.],
    [ 600.,  950., 1300.],
])
b = np.array([148., 162., 201.])

print('Matriz A (usuarios por canal):')
print(A)
print('\nVector b (activaciones totales):', b)

## 2 — Eliminación gaussiana paso a paso

**Operaciones elementales de fila** (no cambian el conjunto solución):
1. **Swap:** $R_i \leftrightarrow R_j$
2. **Scale:** $R_i \leftarrow c \cdot R_i$
3. **Axpy:** $R_i \leftarrow R_i + c \cdot R_j$

Meta: convertir la **matriz aumentada** $[A \mid \mathbf{b}]$ en forma escalonada reducida.

In [ ]:
def gauss_elim(A, b):
    """Eliminación gaussiana con sustitución hacia atrás. Devuelve x."""
    n = len(b)
    Aug = np.hstack([A.copy().astype(float), b.copy().reshape(-1,1).astype(float)])
    print('Matriz aumentada inicial:')
    print(Aug)

    for col in range(n):
        # Pivoteo parcial
        max_row = np.argmax(np.abs(Aug[col:, col])) + col
        Aug[[col, max_row]] = Aug[[max_row, col]]

        for row in range(col+1, n):
            factor = Aug[row, col] / Aug[col, col]
            Aug[row] = Aug[row] - factor * Aug[col]

    print('\nForma escalonada:')
    print(Aug.round(4))

    # Sustitución hacia atrás
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (Aug[i, -1] - np.dot(Aug[i, i+1:n], x[i+1:])) / Aug[i, i]

    return x

x_manual = gauss_elim(A, b)
print('\nSolución (tasas de conversión base):', x_manual.round(6))
print('Verificación A@x ≈ b:', np.allclose(A @ x_manual, b))

## 3 — `np.linalg.solve` (más estable y rápido)

`np.linalg.solve(A, b)` resuelve $A\mathbf{x} = \mathbf{b}$ usando factorización LU con pivoteo parcial (LAPACK `dgesv`). Es **numéricamente más estable** que invertir $A$.

In [ ]:
x_numpy = np.linalg.solve(A, b)
print('Solución con np.linalg.solve:', x_numpy.round(6))
print('¿Coincide con método manual?', np.allclose(x_manual, x_numpy))

print('\n--- Interpretación ---')
for canal, tasa in zip(['Organic', 'Paid', 'Email'], x_numpy):
    print(f'  {canal}: tasa de conversión base = {tasa*100:.4f}%')

## 4 — Los tres casos posibles

| Caso | Condición | Solución |
|---|---|---|
| Solución única | $A$ es cuadrada y $\det(A) \neq 0$ | Un punto |
| Sin solución (inconsistente) | Las ecuaciones se contradicen | $\emptyset$ |
| Infinitas soluciones | Ecuaciones redundantes (rango deficiente) | Un subespacio |

In [ ]:
# Caso 1: Solución única (ya resuelto)
print('CASO 1 - Solución única:')
print('  det(A) =', np.linalg.det(A).round(2))
print('  rango  =', np.linalg.matrix_rank(A))

# Caso 2: Sin solución (sistema inconsistente)
A2 = np.array([[1., 2.], [2., 4.]])
b2_inc = np.array([3., 7.])  # 2*fila1 = [2,4] pero b2[1]=7 ≠ 6
print('\nCASO 2 - Inconsistente:')
print('  det(A2) =', np.linalg.det(A2).round(4), '(≈0 → singular)')
try:
    np.linalg.solve(A2, b2_inc)
except np.linalg.LinAlgError as e:
    print('  Error esperado:', e)

# Caso 3: Infinitas soluciones
b2_dep = np.array([3., 6.])   # fila2 = 2×fila1, b2[1] = 2×b2[0] → consistente pero redundante
print('\nCASO 3 - Infinitas soluciones:')
print('  A2 tiene filas dependientes: fila2 = 2×fila1')
print('  Hay infinitos (x1, x2) tales que x1 + 2*x2 = 3')

## 5 — Visualización geométrica: 2 ecuaciones, 2 incógnitas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x_range = np.linspace(-1, 5, 200)

configs = [
    {'title': 'Solución única\n(intersección en un punto)',
     'lines': [(1, -0.5, 1, '#4361ee', 'x+2y=3... → y=(3-x)/2'),
               (-2, 1, -1, '#f72585', '2x+y=5... → y=5-2x')],
     'sol': (7/3, 1/3)},
    {'title': 'Sin solución\n(rectas paralelas)',
     'lines': [(1, -0.5, 1, '#4361ee', 'x+2y=3'),
               (1, -0.5, 1.8, '#f72585', 'x+2y=7 (paralela)')],
     'sol': None},
    {'title': 'Infinitas soluciones\n(rectas coincidentes)',
     'lines': [(1, -0.5, 1, '#4361ee', 'x+2y=3'),
               (1, -0.5, 1, '#f72585', '2x+4y=6 (misma recta)', '--')],
     'sol': 'linea'},
]

for ax, cfg in zip(axes, configs):
    for item in cfg['lines']:
        a_, b_, c_, col, lbl = item[0], item[1], item[2], item[3], item[4]
        ls = item[5] if len(item) > 5 else '-'
        # a*x + b*y = c → y = (c - a*x)/b  (b aqui es coef de y)
        y_line = (c_ - a_*x_range) / (b_ if b_ != 0 else 1e-9)
        # solo pintar si b_ != 0 (no vertical)
        if abs(b_) > 1e-6:
            ax.plot(x_range, y_line, color=col, lw=2, ls=ls, label=lbl)
    if cfg['sol'] and cfg['sol'] != 'linea':
        ax.plot(*cfg['sol'], 'o', color='#ff9f1c', ms=10, zorder=5, label='Solución')
    ax.set_xlim(-1, 5); ax.set_ylim(-1, 4)
    ax.axhline(0, color='#ccc', lw=0.8); ax.axvline(0, color='#ccc', lw=0.8)
    ax.set_title(cfg['title'], fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Geometría de sistemas lineales 2×2', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Resumen

| Concepto | Descripción | NumPy |
|---|---|---|
| Forma matricial | $A\mathbf{x} = \mathbf{b}$ | Representación estándar |
| Eliminación gaussiana | Reduce a forma escalonada | Implementación manual |
| Solución numérica | Factorización LU con pivoteo | `np.linalg.solve(A, b)` |
| Rango | N° de filas/columnas linealmente independientes | `np.linalg.matrix_rank(A)` |
| Determinante | ≠ 0 ↔ solución única | `np.linalg.det(A)` |

**Siguiente:** `04_matrix_multiplication.ipynb`